# Collect model inputs

Pulls aligned behavior, firing rates and ICA ensembles for the selected sessions from the lab NAS
and writes them to `outputs/glm_input_data/`, which is where `utils/generate_split.py`,
`training/train_*.py` and the `eval/` scripts read from.

**Before running**
- The NAS must be reachable (VPN on / network drive mounted).
- Your OS user must have an entry in `baseVR/base_functionality.py::device_paths()` pointing at the NAS.
- `baseVR`, `ephysVR` and `analysisVR` must sit side by side in the same parent folder.
- Run this notebook from `embedding_development/utils/` (the default when opened in Jupyter/VS Code).

Then run all cells top to bottom. Downloading is slow over VPN (~30 min).

**Outputs** (`outputs/glm_input_data/`)

| File | Contents |
|---|---|
| `behavior_glm_input.npy` (+ `_index`, `_columns`) | All `Behavior40msAligned` columns per 40 ms bin (training scripts pick the features they use by name) |
| `fr_full.npy` (+ `_index`, `_columns`) | Single-unit firing rates (Hz) per 40 ms bin, NaN → 0 |
| `ensembles.npy` | ICA ensemble weight matrix (neurons × ensembles) |


In [ ]:
import os
import sys

# add the project root (parent of analysisVR/, baseVR/, ephysVR/) to sys.path
sys.path.append(os.path.join(os.getcwd(), "..", "..", ".."))
from baseVR.base_functionality import init_import_paths
init_import_paths()

In [ ]:
import numpy as np
import pandas as pd

from analytics_processing import analytics
from CustomLogger import CustomLogger as Logger
from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args
from analytics_processing.sessions_from_nas_parsing import fullfnames2snames

Logger().init_logger(None, None, logging_level="WARNING")

## Settings

In [ ]:
animal_ids = [6]
paradigm_ids = [1100]
session_ids = None  # None = all sessions of this animal/paradigm

output_dir = os.path.abspath(os.path.join("..", "outputs", "glm_input_data"))
os.makedirs(output_dir, exist_ok=True)
print("Writing to:", output_dir)

## Load from NAS

In [ ]:
session_dirs, _ = sessionlist_fullfnames_from_args(paradigm_ids, animal_ids, session_ids)
session_names = fullfnames2snames(session_dirs)
print(f"{len(session_names)} sessions found")

In [ ]:
# Firing rates: one row per 40 ms bin, index = (session_id, session_t, global_t)
fr = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
fr = fr.drop("from_ephys_timestamp", axis=1)
fr.index = fr.index.droplevel((0, 1, 3))
fr.set_index('to_ephys_timestamp', append=True, inplace=True)
fr['global_t'] = np.arange(40_000, 40_000*len(fr)+1, 40_000)
fr.set_index('global_t', append=True, inplace=True)
fr.index = fr.index.rename(['session_id', 'session_t', 'global_t'])
fr = fr.fillna(0)
print("firing rates:", fr.shape)

In [ ]:
# Behavior: all aligned columns; feature selection happens in the training scripts
behavior_glm_input = analytics.get_analytics('Behavior40msAligned', session_names=session_names)
behavior_glm_input.index = behavior_glm_input.index.droplevel(['paradigm_id', 'animal_id'])
print("behavior:", behavior_glm_input.shape)
print("NaN fraction per column:")
print((behavior_glm_input.isna().mean()).round(3).to_string())

In [ ]:
# ICA ensemble weights (neurons x ensembles)
ensembles = analytics.get_analytics('ConcatenatedEnsambles40ms', session_names=session_names)
print("ensembles:", ensembles.shape)

## Sanity checks and save

In [ ]:
assert len(fr) == len(behavior_glm_input), \
    f"row count mismatch: firing rates {len(fr)} vs behavior {len(behavior_glm_input)}"
assert ensembles.shape[0] == fr.shape[1], \
    f"ensemble weights cover {ensembles.shape[0]} neurons but there are {fr.shape[1]} units"

def save(name, df):
    np.save(os.path.join(output_dir, f"{name}.npy"), df.values)
    np.save(os.path.join(output_dir, f"{name}_index.npy"), df.index.values)
    np.save(os.path.join(output_dir, f"{name}_columns.npy"), np.array(df.columns, dtype=object))
    print(f"saved {name} {df.shape}")

save("behavior_glm_input", behavior_glm_input)
save("fr_full", fr)
np.save(os.path.join(output_dir, "ensembles.npy"), ensembles.values)
print(f"saved ensembles {ensembles.shape}")
print("\nDone. Files in", output_dir)